In [ ]:
!pip install datasets pandas -q

In [ ]:
# Colab 셀 2: 설정 및 포맷팅 함수 정의
from datasets import load_dataset, concatenate_datasets
import os

### 설정 값 ###
TOTAL_SIZE = 10000
FINANCE_RATIO = 0.8
OUTPUT_FILENAME = "dpo_dataset.jsonl"
### 설정 끝 ###

def format_safety(example):
    """ko_Ultrafeedback_binarized 데이터셋을 DPO 형식으로 변환합니다."""
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

def format_won_instruct(example):
    """KRX-Data/Won-Instruct 데이터셋을 DPO 형식으로 변환합니다."""
    return {
        "prompt": example['question'],
        "chosen": example['answer_B'],
        "rejected": example['answer_A'],
    }

print("✅ 설정 및 함수 정의 완료!")

✅ 설정 및 함수 정의 완료!


In [ ]:
# Colab 셀 3: 데이터셋 로드
# 금융 데이터 (Won-Instruct) 로드
finance_ds_raw = load_dataset("aiqwe/FinShibainu", name='qa' ,split="train")

# 일반 데이터 (ko_Ultrafeedback) 로드
general_ds_raw = load_dataset("sssssungjae/translated_safety_dpo_ko_fixed_final", split="train")

print("--- 금융 데이터 (Won-Instruct) 정보 ---")
print(finance_ds_raw)
print("\n--- 일반 데이터 (ko_Ultrafeedback) 정보 ---")
print(general_ds_raw)

print("\n\n✅ 데이터셋 로드 완료!")

--- 금융 데이터 (Won-Instruct) 정보 ---
Dataset({
    features: ['reference', 'question', 'answer_A', 'answer_B', 'preference', 'preference_desc', 'value', 'type'],
    num_rows: 44870
})

--- 일반 데이터 (ko_Ultrafeedback) 정보 ---
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 4892
})


✅ 데이터셋 로드 완료!


In [ ]:
# Colab 셀 4: 데이터 전처리 및 형식 변환
# 각 데이터셋에서 가져올 샘플 수 계산
finance_count = int(TOTAL_SIZE * FINANCE_RATIO)
general_count = TOTAL_SIZE - finance_count

# 금융 데이터 전처리
finance_dpo = finance_ds_raw.map(
    format_won_instruct,
    remove_columns=finance_ds_raw.column_names
).shuffle(seed=42).select(range(finance_count))

# 일반 데이터 전처리
general_dpo = general_ds_raw.map(
    format_safety,
    remove_columns=general_ds_raw.column_names
).shuffle(seed=42).select(range(general_count))


print("--- 변환된 금융 데이터 샘플 ---")
print(finance_dpo[0])
print("\n--- 변환된 일반 데이터 샘플 ---")
print(general_dpo[0])
print("\n\n✅ DPO 형식으로 데이터 변환 완료!")

--- 변환된 금융 데이터 샘플 ---
{'prompt': '예치환거래은행에 대한 규제는 어떻게 이루어지나요?', 'chosen': '예치환 거래은행(Deposit Bank) 혹은 예금을 취급하는 은행에 대한 규제는 여러 가지 요소에 따라 다르게 이루어집니다. 이러한 규제는 기본적으로 금융의 안정성과 소비자 보호, 그리고 경제의 전체적인 건전성을 유지하기 위해 설정됩니다. 다음은 주요 규제 요소를 정리한 내용입니다:\n\n1. **금융당국의 감독**: 각국의 중앙은행이나 금융감독기관이 예치환거래은행을 감독합니다. 예를 들어, 미국의 경우 연방준비제도(Federal Reserve)와 금융청(OCC) 등이 해당 역할을 맡고 있습니다. 이러한 기관들은 은행의 자본 적정성, 유동성, 자산 건전성 등을 감독합니다.\n\n2. **자본 비율 규제**: 은행은 최소 자본 비율을 유지해야 하며, 이는 은행이 직면할 수 있는 손실을 감당할 수 있도록 하기 위함입니다. 바젤 III와 같은 국제적인 기준에 따르면, 예치환 거래은행은 위험가중 자산(RWA)에 따라 최소 4%의 기본 자본 비율과 8%의 총 자본 비율을 유지해야 합니다.\n\n3. **예금자 보호**: 많은 국가에서 예금자 보호제도(DIC, FDIC 등)가 운영되어 예금자들이 일정 금액 (예: 미국의 FDIC는 25만 달러까지)까지 보호받을 수 있도록 합니다. 이는 고객의 신뢰를 높이고 은행 시스템의 안정성을 보장하는 역할을 합니다.\n\n4. **대출 및 투자 제한**: 예치환 거래은행은 대출 및 투자에 대한 규제를 받으며, 이는 은행이 불필요한 위험을 감수하지 않도록 하기 위함입니다. 예를 들어, 대출 포트폴리오의 위험 분산이 필요하며, 원화 대출 비율, 외환 노출 등에 대한 제한이 있을 수 있습니다.\n\n5. **보고 및 투명성 의무**: 은행은 정기적으로 재무상태표, 손익계산서와 같은 다양한 재무 정보를 외부에 보고해야 하며, 이 정보는 투자자들과 감독기관이 은행의 재무 건전성을 평가할 수 있도록 돕습니다.\

In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

# --- 사전 준비 ---
# 사용하고 계신 베이스 모델의 ID를 입력해주세요.
model_name = "sssssungjae/qwen2_5-7b-instruct-finance-full-final-15_15"


# 1. 모델의 토크나이저를 로드합니다.
# 이 토크나이저가 대화 형식을 문자열로 변환하는 방법을 알고 있습니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)


# 2. prompt, chosen, rejected 컬럼의 형식을 모두 문자열로 통일하는 함수를 정의합니다.
def unify_dpo_format(example):
    # prompt가 대화형 리스트이면 문자열로 변환
    if isinstance(example['prompt'], list):
        example['prompt'] = tokenizer.apply_chat_template(
            example['prompt'],
            tokenize=False,
            add_generation_prompt=True
        )

    # chosen이 딕셔너리이면 'content' 값만 추출하여 문자열로 변환
    if isinstance(example['chosen'], dict):
        example['chosen'] = example['chosen']['content']

    # rejected가 딕셔너리이면 'content' 값만 추출하여 문자열로 변환
    if isinstance(example['rejected'], dict):
        example['rejected'] = example['rejected']['content']

    return example

# 3. 단일 턴 prompt를 챗 템플릿 형식으로 변환하는 함수를 정의합니다.
def apply_chat_template(example):
    """단순 문자열 prompt를 Qwen 챗 템플릿이 적용된 문자열로 변환합니다."""

    # 단순 문자열을 대화 형식 [{'role': 'user', 'content': '...'}]으로 변환
    chat_format = [{"role": "user", "content": example['prompt']}]

    # 토크나이저의 템플릿을 적용합니다.
    # add_generation_prompt=True는 프롬프트 끝에 assistant의 답변을 유도하는
    # <|im_start|>assistant\n 와 같은 템플릿을 추가해줍니다. (DPO에 필수)
    example['prompt'] = tokenizer.apply_chat_template(
        chat_format,
        tokenize=False,
        add_generation_prompt=True
    )
    return example

# 4. 형식이 다른 데이터셋에 위 함수를 적용하여 모든 컬럼을 통일합니다.
general_dpo_formatted = general_dpo.map(unify_dpo_format)
finance_dpo_formatted = finance_dpo.map(apply_chat_template)

# --- 결과 확인 ---
print("--- 변환된 금융 데이터셋의 첫 번째 샘플 (apply_chat_template 적용 후) ---")
print(finance_dpo_formatted[0])


# 4. 이제 모든 관련 컬럼의 형식이 동일해졌으므로, 에러 없이 병합할 수 있습니다.
final_dpo_dataset = concatenate_datasets([finance_dpo_formatted, general_dpo_formatted]).shuffle(seed=42)

print("--- 최종 데이터셋 정보 ---")
print(final_dpo_dataset)
print("\n--- 변환된 데이터셋의 첫 번째 샘플 ---")
# 모든 컬럼이 문자열로 통일되었는지 확인
print(final_dpo_dataset[0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

--- 변환된 금융 데이터셋의 첫 번째 샘플 (apply_chat_template 적용 후) ---
{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n예치환거래은행에 대한 규제는 어떻게 이루어지나요?<|im_end|>\n<|im_start|>assistant\n', 'chosen': '예치환 거래은행(Deposit Bank) 혹은 예금을 취급하는 은행에 대한 규제는 여러 가지 요소에 따라 다르게 이루어집니다. 이러한 규제는 기본적으로 금융의 안정성과 소비자 보호, 그리고 경제의 전체적인 건전성을 유지하기 위해 설정됩니다. 다음은 주요 규제 요소를 정리한 내용입니다:\n\n1. **금융당국의 감독**: 각국의 중앙은행이나 금융감독기관이 예치환거래은행을 감독합니다. 예를 들어, 미국의 경우 연방준비제도(Federal Reserve)와 금융청(OCC) 등이 해당 역할을 맡고 있습니다. 이러한 기관들은 은행의 자본 적정성, 유동성, 자산 건전성 등을 감독합니다.\n\n2. **자본 비율 규제**: 은행은 최소 자본 비율을 유지해야 하며, 이는 은행이 직면할 수 있는 손실을 감당할 수 있도록 하기 위함입니다. 바젤 III와 같은 국제적인 기준에 따르면, 예치환 거래은행은 위험가중 자산(RWA)에 따라 최소 4%의 기본 자본 비율과 8%의 총 자본 비율을 유지해야 합니다.\n\n3. **예금자 보호**: 많은 국가에서 예금자 보호제도(DIC, FDIC 등)가 운영되어 예금자들이 일정 금액 (예: 미국의 FDIC는 25만 달러까지)까지 보호받을 수 있도록 합니다. 이는 고객의 신뢰를 높이고 은행 시스템의 안정성을 보장하는 역할을 합니다.\n\n4. **대출 및 투자 제한**: 예치환 거래은행은 대출 및 투자에 대한 규제를 받으며, 이는 은행이 불필요한 위험을 감수하지 않도록 하기 위함입니다. 

In [ ]:
# Colab 셀 8: 데이터셋을 허깅페이스 Hub에 업로드

# ⚠️ 여기를 수정해주세요!
# 본인의 허깅페이스 사용자 이름과 원하는 데이터셋 이름으로 설정하세요.
repo_name = "sssssungjae/dpo_shiba_safety1"

print(f"'{repo_name}' 리포지토리에 데이터셋을 업로드합니다...")

# Case 1: 공개(public) 리포지토리로 업로드할 경우
final_dpo_dataset.push_to_hub(repo_name)

# Case 2: 비공개(private) 리포지토리로 업로드할 경우 (둘 중 하나만 사용)
# final_dpo_dataset.push_to_hub(repo_name, private=True)


print("-" * 30)
print(f"🎉 업로드 완료! https://huggingface.co/datasets/{repo_name} 에서 확인하세요.")

'sssssungjae/dpo_shiba_safety1' 리포지토리에 데이터셋을 업로드합니다...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  85%|########4 | 11.8MB / 13.9MB            

------------------------------
🎉 업로드 완료! https://huggingface.co/datasets/sssssungjae/dpo_shiba_safety1 에서 확인하세요.
